# Processar casos confirmados de dengue — POA (direto dos zips do SINAN)

Lê os arquivos nacionais brutos do SINAN (`DENGBR*.csv.zip` em `bases_governo/`), filtra **Porto Alegre** (`ID_MUNICIP=431490`) e **casos confirmados** (`CLASSI_FIN ∈ {10,11,12}`), e gera um CSV único por caso: `output/casos_confirmados_poa.csv`.

Lê **direto dos `.zip`** (não precisa descompactar nem da pasta `csvs/`). Para **atualizar**, baixe um zip novo do OpenDataSUS, substitua na pasta e rode o notebook de novo. Substitui o `base_oficial_filtrada_poa/consolidado` (que cobria só 2020–2025, dos arquivos antigos).

**Granularidade: por caso** — preserva `SEM_PRI`, `CLASSI_FIN`, `EVOLUCAO`, `HOSPITALIZ`, `DT_OBITO`, `SOROTIPO`… A agregação semanal (por `SEM_PRI`) fica no `modelo1.ipynb`.

## Bloco 1 — localizar os arquivos brutos

In [ ]:
import glob
import os
import zipfile
from pathlib import Path

import pandas as pd


# Acha a raiz do projeto (Meu_Projeto/) subindo até encontrar a pasta 'Raspagem'.
def achar_raiz(marcador="Raspagem"):
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / marcador).is_dir():
            return p
    raise FileNotFoundError(f"Pasta-raiz contendo '{marcador}/' nao encontrada a partir de {Path.cwd()}")


RAIZ = achar_raiz()
BASES_GOVERNO = RAIZ / "Bases de dados" / "bases_governo"

zips = sorted(glob.glob(str(BASES_GOVERNO / "DENGBR*.csv.zip")))
print(f"{len(zips)} arquivos:")
for z in zips:
    print("  ", os.path.basename(z), f"({round(os.path.getsize(z) / 1e6, 1)} MB)")

## Bloco 2 — filtrar POA confirmados de cada arquivo

Para cada zip: detecta o separador, lê **em chunks** só as colunas úteis e mantém as linhas de POA (`ID_MUNICIP=431490`) com dengue confirmada (`CLASSI_FIN ∈ {10,11,12}`). (O `DENGBR24` é grande — esta célula leva ~2–4 min no total.)

In [ ]:
# colunas úteis (subconjunto do schema SINAN) que queremos preservar por caso
COLUNAS = ["SEM_PRI", "SEM_NOT", "DT_SIN_PRI", "DT_NOTIFIC", "NU_ANO", "ID_MUNICIP", "ID_MN_RESI",
           "CLASSI_FIN", "CRITERIO", "EVOLUCAO", "DT_OBITO", "HOSPITALIZ", "CS_SEXO", "NU_IDADE_N",
           "CS_GESTANT", "CS_RACA", "SOROTIPO"]


def detectar_separador(zip_path):
    """Lê a 1a linha do CSV dentro do zip e decide entre ',' e ';'."""
    with zipfile.ZipFile(zip_path) as z:
        nome = [n for n in z.namelist() if n.lower().endswith(".csv")][0]
        with z.open(nome) as f:
            primeira = f.readline().decode("latin1")
    sep = ";" if primeira.count(";") > primeira.count(",") else ","
    return sep, primeira.strip().split(sep)


def filtrar_poa_confirmados(zip_path):
    sep, cols = detectar_separador(zip_path)
    usar = [c for c in COLUNAS if c in cols]  # só as colunas que existem nesse ano
    partes = []
    for chunk in pd.read_csv(zip_path, sep=sep, encoding="latin1", usecols=usar,
                             chunksize=300_000, low_memory=False):
        poa = chunk[(pd.to_numeric(chunk["ID_MUNICIP"], errors="coerce") == 431490) &
                    (pd.to_numeric(chunk["CLASSI_FIN"], errors="coerce").isin([10, 11, 12]))]
        partes.append(poa)
    df = pd.concat(partes, ignore_index=True)
    df["arquivo_origem"] = os.path.basename(zip_path)
    return df.reindex(columns=COLUNAS + ["arquivo_origem"])


frames = []
for zp in zips:
    df = filtrar_poa_confirmados(zp)
    print(f"{os.path.basename(zp):20s} POA confirmados = {len(df):,}")
    frames.append(df)

## Bloco 3 — concatenar e visão geral

In [ ]:
casos_confirmados_poa = pd.concat(frames, ignore_index=True)

print("total de casos confirmados POA:", len(casos_confirmados_poa))
sem_pri = pd.to_numeric(casos_confirmados_poa["SEM_PRI"], errors="coerce").dropna().astype(int)
print("SEM_PRI:", sem_pri.min(), "->", sem_pri.max(), "| semanas distintas:", sem_pri.nunique())
print("por ano de notificação (NU_ANO):",
      pd.to_numeric(casos_confirmados_poa["NU_ANO"], errors="coerce").value_counts().sort_index().to_dict())
casos_confirmados_poa.head()

## Bloco 4 — salvar `casos_confirmados_poa.csv`

In [ ]:
OUTPUT_DIR = BASES_GOVERNO / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
destino = OUTPUT_DIR / "casos_confirmados_poa.csv"

casos_confirmados_poa.to_csv(destino, index=False)
print("salvo em:", destino)
print("linhas x colunas:", casos_confirmados_poa.shape)
print("tamanho:", round(destino.stat().st_size / 1_000_000, 2), "MB")